In [0]:
%run ../00_setup/01_config

In [0]:
# Databricks notebook source
# 01_bronze_ingestion_framework.py
# Generic Bronze ingestion function — handles all 12 tables

import uuid
import hashlib
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# Import config (assumes 01_config is run/imported in the same session)
# %run ../00_setup/01_config

def ingest_table(table_name: str, day_number: int, batch_id: str = None):
    """
    Ingests a single table's CSV file for a given day into Bronze.
    Skips gracefully if the table isn't expected on this day.
    """

    run_id = str(uuid.uuid4())
    batch_id = batch_id or str(uuid.uuid4())
    started_at = datetime.now()

    # STEP 2 — Check if this table is expected today
    is_static_or_master = table_name in STATIC_TABLES or table_name in MASTER_TABLES
    if is_static_or_master and day_number != 1:
        print(f"Skipping {table_name} — static/master table, not expected on day {day_number}")
        return

    # STEP 1 — Build source path
    source_path = f"{RAW_BASE_PATH}day_{day_number}/{table_name}.csv"

    # STEP 3 — Check file exists
    try:
        dbutils.fs.ls(source_path)
    except Exception as e:
        error_message = f"File not found: {source_path}"
        _write_audit_log(
            run_id, "bronze", table_name, day_number, source_path,
            started_at, datetime.now(), 0, 0, "FAILED", error_message
        )
        print(error_message)
        return

    # STEP 4 — Read CSV as all strings
    df = (
        spark.read
        .option("header", "true")
        .option("delimiter", "|")
        .option("inferSchema", "false")
        .option("rescuedDataColumn", "_rescued_data")
        .csv(source_path)
    )

    # Cast every column to string explicitly (belt and braces)
    for c in df.columns:
        if c != "_rescued_data":
            df = df.withColumn(c, df[c].cast(StringType()))

    # STEP 5 — Capture source schema
    current_columns = [c for c in df.columns if c != "_rescued_data"]

    # STEP 6 — Detect schema changes vs Day 1 baseline
    _check_schema_drift(table_name, day_number, current_columns)

    # STEP 7 — Add metadata columns
    source_cols_concat = F.concat_ws("|", *[F.coalesce(F.col(c), F.lit("")) for c in current_columns])

    df = (
        df.withColumn("_source_file", F.lit(source_path))
          .withColumn("_ingestion_timestamp", F.current_timestamp())
          .withColumn("_day_number", F.lit(day_number))
          .withColumn("_batch_id", F.lit(batch_id))
          .withColumn("_row_hash", F.sha2(source_cols_concat, 256))
    )

    if "_rescued_data" not in df.columns:
        df = df.withColumn("_rescued_data", F.lit(None).cast(StringType()))

    # STEP 8 — Count source records
    source_count = df.count()

    # STEP 9 — Write to Bronze Delta table
    target_table = f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.brz_{table_name}"
    try:
        (
            df.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(target_table)
        )
        target_count = spark.table(target_table).filter(
            (F.col("_day_number") == day_number) & (F.col("_batch_id") == batch_id)
        ).count()
        status = "SUCCESS"
        error_message = None
    except Exception as e:
        target_count = 0
        status = "FAILED"
        error_message = str(e)

    completed_at = datetime.now()

    # STEP 10 — Write audit entry + update control table
    _write_audit_log(
        run_id, "bronze", table_name, day_number, source_path,
        started_at, completed_at, source_count, target_count, status, error_message
    )

    if status == "SUCCESS":
        _update_control_table(table_name, "bronze", day_number)

    print(f"{table_name} (day {day_number}): {status} — {source_count} read, {target_count} written")


def _check_schema_drift(table_name, day_number, current_columns):
    """Compares current columns against Day 1 baseline and logs any drift."""
    baseline_table = f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.brz_{table_name}"

    try:
        existing = spark.table(baseline_table)
        baseline_columns = [c for c in existing.columns if not c.startswith("_")]
    except Exception:
        # No prior data — this IS the baseline, nothing to compare
        return

    added = list(set(current_columns) - set(baseline_columns))
    removed = list(set(baseline_columns) - set(current_columns))

    if added or removed:
        change_row = [(
            table_name,
            day_number,
            datetime.now(),
            baseline_columns,
            current_columns,
            added,
            removed,
            []  # renamed detection would need manual mapping — left empty here
        )]
        columns = [
            "table_name", "day_number", "detected_at",
            "previous_columns", "current_columns",
            "columns_added", "columns_removed", "columns_renamed"
        ]
        change_df = spark.createDataFrame(change_row, columns)
        change_df.write.format("delta").mode("append").saveAsTable(
            f"{DQ_CATALOG}.{DQ_SCHEMA}.schema_change_log"
        )


def _write_audit_log(run_id, pipeline_layer, table_name, day_number, source_path,
                      started_at, completed_at, source_count, target_count, status, error_message):
    # Convert None to empty string to avoid type inference issues
    error_message = error_message if error_message is not None else ""
    
    row = [(
        run_id, pipeline_layer, table_name, day_number, source_path,
        started_at, completed_at, source_count, target_count, status, error_message
    )]
    columns = [
        "run_id", "pipeline_layer", "table_name", "day_number", "source_path",
        "started_at", "completed_at", "source_record_count", "target_record_count",
        "status", "error_message"
    ]
    audit_df = spark.createDataFrame(row, columns)
    audit_df.write.format("delta").mode("append").saveAsTable(
        f"{DQ_CATALOG}.{DQ_SCHEMA}.pipeline_audit_log"
    )


def _update_control_table(table_name, pipeline_layer, day_number):
    control_table = f"{DQ_CATALOG}.{DQ_SCHEMA}.pipeline_control"
    exists = spark.sql(f"""
        SELECT COUNT(*) as cnt FROM {control_table}
        WHERE table_name = '{table_name}' AND pipeline_layer = '{pipeline_layer}'
    """).collect()[0]["cnt"] > 0

    if exists:
        spark.sql(f"""
            UPDATE {control_table}
            SET last_successful_day = {day_number},
                last_run_at = current_timestamp(),
                total_runs = total_runs + 1
            WHERE table_name = '{table_name}' AND pipeline_layer = '{pipeline_layer}'
        """)
    else:
        row = [(table_name, pipeline_layer, day_number, datetime.now(), 1)]
        columns = ["table_name", "pipeline_layer", "last_successful_day", "last_run_at", "total_runs"]
        spark.createDataFrame(row, columns).write.format("delta").mode("append").saveAsTable(control_table)